# Phase G: Ablations and generalisation

These are controlled variations of the Phase F monitors. The protocol is the same in every experiment:

- logistic regression;
- 5-fold cross-validation grouped by route;
- thresholds chosen on validation routes only;
- results reported at the **2% and 5% alarm-time budgets**, i.e. the share of normal driving spent in alarm.

| Section | Question |
|---|---|
| 1 | **History length:** current frame vs 0.5, 1, 2 and 3 s of temporal features on top of it. Is history itself useful? |
| 2 | **Feature groups:** path, speed distribution, controls, actor geometry, and combinations. Which signals matter? |
| 3 | **Safe corrections:** training with vs without benign "hard negatives" (successful manoeuvres, benign planner instability). Does seeing them reduce false alarms? |
| 4 | **Scenario families:** junctions, merges and detours separately |
| 5 | **Representation:** linear model vs a non-linear one (gradient boosting) on the same handcrafted features, and whether a GRU is justified |

With about 60 collisions, a difference of one collision is about 2 points, and differences under about 12 points are not conclusive.

In [1]:
import sys, csv, json, collections, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
import ew_analysis as ew
import fg_analysis as fg

RESULTS = REPO / "results" / "bench2drive_v2"
OUT = RESULTS / "fg"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(rows, cols=None):
    cols = cols or list(rows[0])
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(f"{r[c]:.3f}" if isinstance(r.get(c), float) else str(r.get(c, "")) for c in cols) + " |")
    display(Markdown("\n".join(out)))

def save_csv(name, rows):
    with open(OUT / f"{name}.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

p = lambda v: f"{100 * v:.0f}%"
pci = lambda v, c: f"{100 * v:.0f}% [{100 * c[0]:.0f}, {100 * c[1]:.0f}]"

# dataset: one row per 20 Hz step, label = a counted collision within 3 s (built by ew_analysis, unchanged)
routes, ckpts, EW = ew.build_cache(RESULTS, REPO / "carla_garage/leaderboard/data/bench2drive220.xml", RESULTS / "early_warning" / "cache.pkl")
events = ew.event_table(routes, ckpts, EW)
F, M, ev_index, untimed = ew.build_dataset(routes, ckpts, EW, events)
EIDS = sorted({e["eid"] for e in ev_index})
G = M["route"]
print(f"routes {len(np.unique(G))} | steps {len(M['y']):,} | positive steps {M['y'].sum():,} | moving steps {M['moving'].sum():,}")
print(f"collision events {len(EIDS)} | normal driving {(M['y'] == 0).sum() * ew.DT / 60:.0f} min | excluded (off-route only, no timestamp): {untimed}")

def run(cols, kind="lr", train_mask=None):
    score, thr = fg.cross_validate(fg.design(F, cols), M["y"], G, kind=kind, train_mask=train_mask)
    auc, ap = fg.ranking(score, M)
    return score, thr, {"AUROC (moving)": auc, "AUPRC (moving)": ap,
                        **{f"warned >= 1 s @{int(b * 100)}%": fg.alarm_metrics(score, thr[b], M, EIDS)["recall_1s"] for b in (0.02, 0.05)},
                        **{f"warned >= 2 s @{int(b * 100)}%": fg.alarm_metrics(score, thr[b], M, EIDS)["recall_2s"] for b in (0.02, 0.05)},
                        "false-alarm onsets/min @2%": fg.alarm_metrics(score, thr[0.02], M, EIDS)["onsets_per_min"]}
CURRENT = fg.GEOMETRY + fg.CONFIDENCE
fmt_rows = lambda rows: [{k: (p(v) if k.startswith("warned") else v) for k, v in r.items()} for r in rows]

routes 214 | steps 147,531 | positive steps 3,042 | moving steps 43,506
collision events 60 | normal driving 120 min | excluded (off-route only, no timestamp): ['2790', '23658', '23910', '24098', '25896']


## 1. History length

The base is the strongest current-frame model (intent-aware geometry + TF++ confidence). Temporal instability features computed over each window are added on top.

In [2]:
rows = []
for tag in (None, "0.5s", "1s", "2s", "3s"):
    cols = CURRENT + ([] if tag is None else ew.TEMPORAL(tag, tag))
    _, _, m = run(cols)
    rows.append({"history": "current frame only" if tag is None else f"+ {tag} of history", **m})
    print(rows[-1]["history"], "done")
table(fmt_rows(rows)); save_csv("G1_history_length", rows)

current frame only done


+ 0.5s of history done


+ 1s of history done


+ 2s of history done


+ 3s of history done


| history | AUROC (moving) | AUPRC (moving) | warned >= 1 s @2% | warned >= 1 s @5% | warned >= 2 s @2% | warned >= 2 s @5% | false-alarm onsets/min @2% |
|---|---|---|---|---|---|---|---|
| current frame only | 0.749 | 0.223 | 42% | 58% | 12% | 30% | 1.802 |
| + 0.5s of history | 0.772 | 0.214 | 60% | 72% | 20% | 43% | 2.367 |
| + 1s of history | 0.764 | 0.217 | 55% | 65% | 20% | 42% | 2.035 |
| + 2s of history | 0.751 | 0.211 | 47% | 70% | 17% | 40% | 1.669 |
| + 3s of history | 0.740 | 0.209 | 45% | 65% | 20% | 40% | 1.910 |

## 2. Feature groups

Each group is used on its own (plus ego speed), then in combinations.

In [3]:
gr = fg.GROUPS
combos = {**{k: v for k, v in gr.items()},
          "path + speed distribution": gr["path"] + gr["speed distribution"],
          "path + speed + controls (all planner/control signals)": gr["path"] + gr["speed distribution"] + gr["controls"],
          "actor geometry + path": gr["actor geometry"] + gr["path"],
          "actor geometry + speed distribution": gr["actor geometry"] + gr["speed distribution"],
          "all groups": sum(gr.values(), [])}
rows = []
for name, cols in combos.items():
    _, _, m = run(["ego_speed"] + cols)
    rows.append({"features (+ ego speed)": name, **m})
    print(name, "done")
table(fmt_rows(rows)); save_csv("G2_feature_groups", rows)

path done


speed distribution done


controls done


actor geometry done


path + speed distribution done


path + speed + controls (all planner/control signals) done


actor geometry + path done


actor geometry + speed distribution done


all groups done


| features (+ ego speed) | AUROC (moving) | AUPRC (moving) | warned >= 1 s @2% | warned >= 1 s @5% | warned >= 2 s @2% | warned >= 2 s @5% | false-alarm onsets/min @2% |
|---|---|---|---|---|---|---|---|
| path | 0.731 | 0.145 | 32% | 65% | 13% | 33% | 1.304 |
| speed distribution | 0.690 | 0.105 | 28% | 43% | 17% | 28% | 0.955 |
| controls | 0.641 | 0.107 | 28% | 62% | 10% | 35% | 1.462 |
| actor geometry | 0.722 | 0.210 | 42% | 62% | 12% | 23% | 1.503 |
| path + speed distribution | 0.755 | 0.136 | 32% | 65% | 20% | 37% | 1.254 |
| path + speed + controls (all planner/control signals) | 0.747 | 0.137 | 33% | 72% | 18% | 38% | 1.387 |
| actor geometry + path | 0.749 | 0.234 | 47% | 65% | 18% | 33% | 1.412 |
| actor geometry + speed distribution | 0.755 | 0.176 | 33% | 55% | 17% | 30% | 1.611 |
| all groups | 0.770 | 0.236 | 53% | 72% | 18% | 47% | 1.678 |

## 3. Safe corrections: do benign hard negatives reduce false alarms?

**Hard negatives** are normal steps that look risky but are not:

- the 3 s before and 1 s after the peak of each successful junction turn, lane change or bypass;
- any moving step whose path disagreement, target-speed JS or steering variation is above the 95th percentile of normal driving.

The *without* variant drops them from **training only**. Validation and test always use every step, so the comparison is fair.

In [4]:
anchors = ew.maneuver_anchors(routes, EW, events)
HN = fg.hard_negatives(F, M, anchors, None)
print(f"hard-negative steps: {HN.sum():,} ({HN.mean():.1%} of all steps)")
rows = []
for name, cols in (("Handcrafted temporal", fg.temporal()), ("Geometry + temporal", fg.GEOMETRY_TEMPORAL),
                   ("Current frame + temporal", CURRENT + fg.temporal()[1:] + ew.TEMPORAL_GEOM)):
    for variant, mask in (("with hard negatives", None), ("without hard negatives", ~HN)):
        score, thr, m = run(cols, train_mask=mask)
        a = score >= thr[0.02]
        rows.append({"monitor": name, "training": variant, **m,
                     "alarm time on hard negatives @2%": float(a[HN].mean()),
                     "alarm time on other normal steps @2%": float(a[(M["y"] == 0) & ~HN].mean())})
        print(name, variant, "done")
table(fmt_rows(rows)); save_csv("G3_safe_corrections", rows)

hard-negative steps: 10,791 (7.3% of all steps)


Handcrafted temporal with hard negatives done


Handcrafted temporal without hard negatives done


Geometry + temporal with hard negatives done


Geometry + temporal without hard negatives done


Current frame + temporal with hard negatives done


Current frame + temporal without hard negatives done


| monitor | training | AUROC (moving) | AUPRC (moving) | warned >= 1 s @2% | warned >= 1 s @5% | warned >= 2 s @2% | warned >= 2 s @5% | false-alarm onsets/min @2% | alarm time on hard negatives @2% | alarm time on other normal steps @2% |
|---|---|---|---|---|---|---|---|---|---|---|
| Handcrafted temporal | with hard negatives | 0.730 | 0.122 | 32% | 62% | 18% | 35% | 1.545 | 0.181 | 0.005 |
| Handcrafted temporal | without hard negatives | 0.737 | 0.125 | 30% | 53% | 18% | 33% | 1.271 | 0.200 | 0.004 |
| Geometry + temporal | with hard negatives | 0.757 | 0.220 | 52% | 67% | 17% | 38% | 1.968 | 0.119 | 0.007 |
| Geometry + temporal | without hard negatives | 0.760 | 0.172 | 38% | 65% | 17% | 37% | 1.528 | 0.175 | 0.004 |
| Current frame + temporal | with hard negatives | 0.760 | 0.221 | 48% | 65% | 17% | 37% | 1.952 | 0.119 | 0.006 |
| Current frame + temporal | without hard negatives | 0.760 | 0.167 | 33% | 63% | 13% | 38% | 1.428 | 0.177 | 0.004 |

## 4. Scenario families

Each monitor is trained once on all routes. Here it is **evaluated** separately on the routes of each core family, with the same validated threshold. There are 13–18 collisions per family, so per-family numbers are indicative only.

In [5]:
fam = np.array([routes[str(r)]["family"] for r in M["route"]])
rows = []
for name, cols in (("Speed only", fg.SPEED), ("TF++ confidence", fg.CONFIDENCE), ("Handcrafted temporal", fg.temporal()),
                   ("Current-frame geometry (intent-aware)", fg.GEOMETRY), ("Geometry + temporal", fg.GEOMETRY_TEMPORAL),
                   ("Current frame + temporal", CURRENT + fg.temporal()[1:] + ew.TEMPORAL_GEOM)):
    score, thr = fg.cross_validate(fg.design(F, cols), M["y"], G)
    for f in ("junction", "lane_merge", "obstacle_detour"):
        mask = fam == f
        a = fg.alarm_metrics(score, thr[0.02], M, EIDS, mask=mask)
        auc, ap = fg.ranking(score, M, mask=mask)
        rows.append({"family": f, "monitor": name, "collisions": a["events"], "AUROC (moving)": auc,
                     "warned >= 1 s @2%": a["recall_1s"], "warned >= 2 s @2%": a["recall_2s"],
                     "time in alarm (family's normal driving)": f"{100 * a['alarm_time']:.1f}%"})
    print(name, "done")
rows.sort(key=lambda r: (r["family"], r["monitor"]))
table(fmt_rows(rows)); save_csv("G4_scenario_families", rows)

Speed only done


TF++ confidence done


Handcrafted temporal done


Current-frame geometry (intent-aware) done


Geometry + temporal done


Current frame + temporal done


| family | monitor | collisions | AUROC (moving) | warned >= 1 s @2% | warned >= 2 s @2% | time in alarm (family's normal driving) |
|---|---|---|---|---|---|---|
| junction | Current frame + temporal | 13 | 0.812 | 62% | 8% | 1.4% |
| junction | Current-frame geometry (intent-aware) | 13 | 0.710 | 38% | 0% | 1.7% |
| junction | Geometry + temporal | 13 | 0.812 | 62% | 8% | 1.5% |
| junction | Handcrafted temporal | 13 | 0.722 | 15% | 8% | 1.6% |
| junction | Speed only | 13 | 0.414 | 8% | 0% | 0.9% |
| junction | TF++ confidence | 13 | 0.461 | 8% | 0% | 0.7% |
| lane_merge | Current frame + temporal | 18 | 0.771 | 39% | 11% | 1.0% |
| lane_merge | Current-frame geometry (intent-aware) | 18 | 0.710 | 28% | 11% | 1.7% |
| lane_merge | Geometry + temporal | 18 | 0.758 | 44% | 17% | 1.1% |
| lane_merge | Handcrafted temporal | 18 | 0.683 | 33% | 17% | 1.9% |
| lane_merge | Speed only | 18 | 0.602 | 22% | 11% | 3.5% |
| lane_merge | TF++ confidence | 18 | 0.609 | 28% | 11% | 3.5% |
| obstacle_detour | Current frame + temporal | 21 | 0.793 | 57% | 24% | 1.8% |
| obstacle_detour | Current-frame geometry (intent-aware) | 21 | 0.757 | 57% | 19% | 1.4% |
| obstacle_detour | Geometry + temporal | 21 | 0.791 | 57% | 19% | 1.7% |
| obstacle_detour | Handcrafted temporal | 21 | 0.753 | 38% | 24% | 2.1% |
| obstacle_detour | Speed only | 21 | 0.700 | 38% | 14% | 1.3% |
| obstacle_detour | TF++ confidence | 21 | 0.708 | 33% | 19% | 1.5% |

## 5. Representation: linear vs non-linear, and is a GRU justified?

The plan allows a small GRU "if the dataset supports it". Before training one, two checks:

1. Does a **non-linear** model (gradient boosting) on the same handcrafted features beat logistic regression? If extra capacity on these features does not help, a sequence model with even more capacity is unlikely to.
2. Does longer history help? (Section 1.)

In [6]:
rows = []
for name, cols in (("Handcrafted temporal", fg.temporal()), ("Current-frame geometry (intent-aware)", fg.GEOMETRY),
                   ("Current frame + temporal", CURRENT + fg.temporal()[1:] + ew.TEMPORAL_GEOM)):
    for kind, lab in (("lr", "logistic regression"), ("gbm", "gradient boosting")):
        _, _, m = run(cols, kind=kind)
        rows.append({"features": name, "model": lab, **m})
        print(name, lab, "done")
table(fmt_rows(rows)); save_csv("G5_representation", rows)

Handcrafted temporal logistic regression done


Handcrafted temporal gradient boosting done


Current-frame geometry (intent-aware) logistic regression done


Current-frame geometry (intent-aware) gradient boosting done


Current frame + temporal logistic regression done


Current frame + temporal gradient boosting done


| features | model | AUROC (moving) | AUPRC (moving) | warned >= 1 s @2% | warned >= 1 s @5% | warned >= 2 s @2% | warned >= 2 s @5% | false-alarm onsets/min @2% |
|---|---|---|---|---|---|---|---|---|
| Handcrafted temporal | logistic regression | 0.730 | 0.122 | 32% | 62% | 18% | 35% | 1.545 |
| Handcrafted temporal | gradient boosting | 0.747 | 0.162 | 68% | 78% | 47% | 63% | 2.425 |
| Current-frame geometry (intent-aware) | logistic regression | 0.716 | 0.205 | 42% | 62% | 12% | 27% | 1.603 |
| Current-frame geometry (intent-aware) | gradient boosting | 0.759 | 0.313 | 55% | 70% | 27% | 48% | 2.492 |
| Current frame + temporal | logistic regression | 0.760 | 0.221 | 48% | 65% | 17% | 37% | 1.952 |
| Current frame + temporal | gradient boosting | 0.795 | 0.361 | 62% | 78% | 42% | 57% | 3.098 |

### 5b. Does temporal history still add value with the non-linear model?

The model is now gradient boosting. This compares the **current frame (geometry + confidence)** with **current frame + temporal**, including the alarm time and alarm durations actually measured on the test routes, plus a paired route-bootstrap of the difference.

In [7]:
GB = {}
for name, cols in (("GBM: speed only", fg.SPEED), ("GBM: current frame (geometry + confidence)", CURRENT),
                   ("GBM: current frame + temporal", CURRENT + fg.temporal()[1:] + ew.TEMPORAL_GEOM),
                   ("GBM: handcrafted temporal", fg.temporal())):
    GB[name] = fg.cross_validate(fg.design(F, cols), M["y"], G, kind="gbm")
    print(name, "done")
rows = []
for name, (score, thr) in GB.items():
    ci = fg.bootstrap(score, thr, M, EIDS, budgets=(0.02,), B=200)
    a = fg.alarm_metrics(score, thr[0.02], M, EIDS)
    rows.append({"monitor": name, "warned >= 1 s @2%": pci(a["recall_1s"], ci["recall_1s@0.02"]),
                 "warned >= 2 s @2%": pci(a["recall_2s"], ci["recall_2s@0.02"]), "median lead (s)": f"{a['median_lead']:.2f}",
                 "time in alarm (test)": f"{100 * a['alarm_time']:.1f}%", "false-alarm onsets/min": f"{a['onsets_per_min']:.2f}",
                 "mean false alarm (s)": f"{a['mean_false_alarm_s']:.1f}"})
table(rows); save_csv("G5b_gbm_budget2", rows)
rows = []
for a_, b_ in (("GBM: current frame + temporal", "GBM: current frame (geometry + confidence)"),
               ("GBM: handcrafted temporal", "GBM: speed only")):
    for k, (pt, lo, hi) in fg.paired_difference(*GB[a_], *GB[b_], M, EIDS, budgets=(0.02, 0.05)).items():
        rows.append({"A minus B": f"{a_} minus {b_}", "metric": k, "difference": f"{pt:+.3f}", "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                     "excludes 0": "yes" if lo > 0 or hi < 0 else ""})
table(rows); save_csv("G5b_gbm_paired", rows)

GBM: speed only done


GBM: current frame (geometry + confidence) done


GBM: current frame + temporal done


GBM: handcrafted temporal done


| monitor | warned >= 1 s @2% | warned >= 2 s @2% | median lead (s) | time in alarm (test) | false-alarm onsets/min | mean false alarm (s) |
|---|---|---|---|---|---|---|
| GBM: speed only | 35% [23, 47] | 18% [10, 27] | 0.00 | 2.6% | 1.21 | 1.3 |
| GBM: current frame (geometry + confidence) | 62% [48, 73] | 32% [20, 44] | 1.30 | 2.3% | 3.16 | 0.5 |
| GBM: current frame + temporal | 62% [48, 75] | 42% [28, 55] | 1.65 | 2.5% | 3.10 | 0.5 |
| GBM: handcrafted temporal | 68% [55, 80] | 47% [32, 58] | 1.88 | 2.4% | 2.43 | 0.6 |

| A minus B | metric | difference | 95% CI | excludes 0 |
|---|---|---|---|---|
| GBM: current frame + temporal minus GBM: current frame (geometry + confidence) | AUPRC (moving) | +0.024 | [-0.021, +0.069] |  |
| GBM: current frame + temporal minus GBM: current frame (geometry + confidence) | recall >=1 s @ 2% | +0.000 | [-0.067, +0.072] |  |
| GBM: current frame + temporal minus GBM: current frame (geometry + confidence) | recall >=2 s @ 2% | +0.100 | [+0.016, +0.192] | yes |
| GBM: current frame + temporal minus GBM: current frame (geometry + confidence) | recall >=1 s @ 5% | +0.067 | [-0.017, +0.159] |  |
| GBM: current frame + temporal minus GBM: current frame (geometry + confidence) | recall >=2 s @ 5% | +0.100 | [+0.000, +0.188] |  |
| GBM: handcrafted temporal minus GBM: speed only | AUPRC (moving) | +0.066 | [+0.024, +0.112] | yes |
| GBM: handcrafted temporal minus GBM: speed only | recall >=1 s @ 2% | +0.333 | [+0.190, +0.467] | yes |
| GBM: handcrafted temporal minus GBM: speed only | recall >=2 s @ 2% | +0.283 | [+0.144, +0.407] | yes |
| GBM: handcrafted temporal minus GBM: speed only | recall >=1 s @ 5% | +0.283 | [+0.154, +0.425] | yes |
| GBM: handcrafted temporal minus GBM: speed only | recall >=2 s @ 5% | +0.283 | [+0.160, +0.412] | yes |

## Takeaways

Results below are at the 2% alarm-time budget unless stated otherwise. Sections 1–4 are single runs without confidence intervals; with 60 collisions, treat differences under about 12 points as indicative.

**1. History length: short history helps, long history does not**

| History on top of the current frame | Warned ≥1 s / ≥2 s @2% | @5% |
|---|---|---|
| None | 42 / 12% | 58 / 30% |
| 0.5 s | **60 / 20%** | 72 / 43% |
| 1 s | 55 / 20% | 65 / 42% |
| 2 s | 47 / 17% | 70 / 40% |
| 3 s | 45 / 20% | 65 / 40% |

AUROC moves only from 0.75 to 0.77. **The useful history is about 0.5–1 s, not the 1–3 s the proposal assumed.**

**2. Feature groups**

| Group (+ ego speed) | AUROC | AUPRC | Warned ≥1 s @2% |
|---|---|---|---|
| Actor geometry | 0.72 | **0.21** | 42% |
| Path | 0.73 | 0.15 | 32% |
| Speed distribution | 0.69 | 0.11 | 28% |
| Controls | 0.64 | 0.11 | 28% |
| Geometry + path | | | 47% |
| All groups together | 0.77 | 0.24 | **53%** (≥2 s: 18%; at 5%: 72% / 47%) |

Path signals are the most useful *temporal* group; controls add little.

**3. Safe corrections: keep the benign hard negatives in training**

- Hard negatives are 7.3% of steps.
- Dropping them from training cuts false-alarm onsets (1.97 → 1.53 per minute for geometry + temporal). But recall falls from 52% to 38%, and the model spends *more* time alarming on benign manoeuvres (12% → 18% of hard-negative time).
- Even with them included, **most false-alarm time falls on legitimate manoeuvres**: monitors are in alarm 12–18% of hard-negative time against under 1% of other normal driving.

**4. Scenario families** (warned ≥1 s @2%)

| Family | Speed / confidence | Temporal | Geometry | Geometry + temporal or current frame + temporal |
|---|---|---|---|---|
| Junctions (13) | 8% (AUROC below 0.5) | 15% | 38% | **62%** |
| Merges (18) | 22–28% | 33% | 28% | 39–44% |
| Detours (21) | 33–38% | 38% | 57% | 57% |

- At junctions, speed and confidence are worse than chance. Temporal history adds most at junctions, and only in combination with geometry (+3 collisions).
- For detours, geometry alone is enough.

**5. Representation: a non-linear model matters more than history length**

Gradient boosting (GBM) on the *same handcrafted features* beats logistic regression (LR) by a wide margin:

| Features | LR, ≥1 s / ≥2 s @2% | GBM, ≥1 s / ≥2 s @2% |
|---|---|---|
| Handcrafted temporal | 32 / 18% | **68 / 47%** |
| Current frame + temporal | 48 / 17% | 62 / 42% |
| Intent-aware geometry | 42 / 12% | 55 / 27% |

Measured on the test routes, the boosted monitors spend 2.3–2.6% of normal driving in alarm. Their alarms are shorter and more frequent: 2.4–3.2 onsets per minute against 1.5–2.0 for logistic regression.

With boosting, two differences are statistically significant:

- **Handcrafted temporal beats speed alone** by +33 points (≥1 s) and +28 points (≥2 s) at 2%.
- **Temporal history adds +10 points of ≥2 s warnings over the current frame** (CI [+2, +19]). It adds nothing at ≥1 s.

**About the GRU:** none was trained. Non-linear interactions between the existing features already capture most of the gain, and history beyond about 1 s does not help. With 60 collisions, a sequence model would mostly add overfitting risk. The evidence supports a **GRU over about 1 s of the same features only after more failures are generated**, compared against boosting on the handcrafted features.

**Overall answers to the plan's three questions**

1. **Failure mechanism** (Phases B–E): mostly planning.
2. **Predictive information:** TF++'s planned path checked against its own tracked boxes, plus short-term path, speed and control instability. Most warnings arrive 1–2 s ahead.
3. **Added temporal value:** yes, but narrowly. Short history (0.5–1 s) consistently adds **≥2 s warning lead time** on top of current-frame geometry and confidence (+10 points, significant with boosting and at the 5% budget with logistic regression). It does not reliably improve ≥1 s recall or precision. The bigger gains come from intent-aware geometry and from non-linear modelling.
